# ModelProject 1.3

1.1 The problem in nested budget shares

In [15]:
from types import SimpleNamespace

import numpy as np

from scipy import optimize

class ConsumerClass:
    """ a consumer with nested CES preferences over three goods

    Good 1 is food. Goods 2 and 3 are bus trips and train trips, and they sit
    together in a nest.

    The problem is written in *nested* budget shares, in the same two steps as
    the nests themselves:

        s1 = the share of income spent on food
        w  = the share of the remaining (travel) budget spent on the bus

    so that s2 = (1-s1)*w and s3 = (1-s1)*(1-w). Any (s1,w) in the unit square
    is a possible choice, and every possible choice is in the unit square, so
    the constraint set is exactly the box that L-BFGS-B takes as `bounds`.

    """

    def __init__(self,par=None):

        # a. setup
        self.setup()

        # b. update parameters
        if not par is None:
            for k,v in par.items():
                self.par.__dict__[k] = v

    def setup(self):
        """ set the baseline parameters """

        par = self.par = SimpleNamespace()
        sol = self.sol = SimpleNamespace()

        # a. preference weights
        par.alpha = 0.60 # weight on food
        par.beta = 0.50 # weight on the bus

        # b. substitution
        par.sigma_A = 0.80 # between food and travel (upper nest)
        par.sigma_B = 0.40 # between bus and train (lower nest)

        # c. prices and income
        par.p1 = 1.0 # price of food
        par.p2 = 1.0 # price of a bus trip
        par.p3 = 1.5 # price of a train trip
        par.I = 10.0 # income

        # d. numerical settings
        par.s_min = 1e-12 # smallest quantity allowed in .ces(), see the note there

    def __str__(self):
        """ print the parameters """

        par = self.par

        lines = ['ConsumerClass']
        lines.append(f'  alpha = {par.alpha:.4f}, beta = {par.beta:.4f}')
        lines.append(f'  sigma_A = {par.sigma_A:.4f}, sigma_B = {par.sigma_B:.4f}')
        lines.append(f'  p1 = {par.p1:.4f}, p2 = {par.p2:.4f}, p3 = {par.p3:.4f}')
        lines.append(f'  I = {par.I:.4f}')

        return '\n'.join(lines)

    ###################
    # 1. the CES nest #
    ###################

    def ces(self,z1,z2,w,sigma):
        """ the CES aggregate of two inputs

        Computes (w*z1**rho + (1-w)*z2**rho)**(1/rho) with rho = 1-1/sigma.

        The inputs are floored at par.s_min, because z**rho is not defined for
        z = 0 when rho < 0, and the corners of the unit square do give zeros.

        Note on the size of par.s_min. It must be *far* below the step that
        L-BFGS-B uses to estimate the gradient, which is `eps` and about 1e-8 by
        default. Otherwise both z and z+eps get floored to the same number near a
        bound, utility comes out exactly equal in the two points, the estimated
        gradient is zero, and the solver stops on the bound and stays there. With
        s_min = 1e-8 that really happens: the answers in section 4 for high tax
        rates come out as zero revenue. 1e-12 is small enough to be invisible and
        large enough to keep z**rho from overflowing.

        Args:

            z1 (float or ndarray): first input
            z2 (float or ndarray): second input
            w (float): weight on the first input
            sigma (float): substitution parameter, must not be 1

        Returns:

            (float or ndarray): the CES aggregate

        """

        par = self.par

        assert not np.isclose(sigma,1.0), 'sigma = 1 gives rho = 0 and a division by zero'

        z1 = np.maximum(z1,par.s_min)
        z2 = np.maximum(z2,par.s_min)

        rho = 1-1/sigma

        return (w*z1**rho + (1-w)*z2**rho)**(1/rho)

    def utility(self,x1,x2,x3):
        """ nested CES utility of a bundle of quantities

        Two steps: first combine goods 2 and 3 into the travel composite, then
        combine good 1 and the composite into utility. Use .ces() for both.

        Args:

            x1 (float or ndarray): quantity of good 1
            x2 (float or ndarray): quantity of good 2
            x3 (float or ndarray): quantity of good 3

        Returns:

            (float or ndarray): utility

        """

        par = self.par

        xB = self.ces(x2, x3, par.beta, par.sigma_B)
        
        u = self.ces(x1, xB, par.alpha, par.sigma_A)

        return u

    ###############################
    # 2. the nested budget shares #
    ###############################

    def shares(self,s1,w):
        """ the three budget shares implied by the nested shares

        Args:

            s1 (float or ndarray): share of income spent on food
            w (float or ndarray): share of the travel budget spent on the bus

        Returns:

            (tuple): the three budget shares, which always sum to one

        """

        return s1,(1-s1)*w,(1-s1)*(1-w)

    def quantities(self,s1,w):
        """ the quantities implied by the nested shares

        Args:

            s1 (float or ndarray): share of income spent on food
            w (float or ndarray): share of the travel budget spent on the bus

        Returns:

            (tuple): the three quantities

        """

        par = self.par

        s1,s2,s3 = self.shares(s1,w)

        return s1*par.I/par.p1, s2*par.I/par.p2, s3*par.I/par.p3

    def value_of_choice(self,s1,w):
        """ utility of the bundle implied by the nested shares

        Args:

            s1 (float or ndarray): share of income spent on food
            w (float or ndarray): share of the travel budget spent on the bus

        Returns:

            (float or ndarray): utility

        """

        x1, x2, x3 = self.quantities(s1, w)
        u = self.utility(x1, x2, x3)

        return u

    def objective(self,s):
        """ minus utility, for a minimizer

        Nothing else is needed: the bounds are the whole constraint.

        Args:

            s (ndarray): array of length 2 with (s1,w)

        Returns:

            (float): minus utility

        """

        return -self.value_of_choice(s[0],s[1])

    #################
    # 3. solving it #
    #################

    def solve_grid(self,N=200,do_print=True):
        """ solve by a 2-dimensional grid search over the nested shares

        Every point of the unit square is a possible choice, so there is nothing
        to mask out here -- a plain np.argmax will do.

        Args:

            N (int): number of grid points for each variable
            do_print (bool): print the solution

        Returns:

            (SimpleNamespace): the grids, the utility values and the best point

        """

        par = self.par
        opt = SimpleNamespace()

        # a. the two grids
        s1_vec = np.linspace(0, 1, N)
        w_vec = np.linspace(0, 1, N)

        opt.s1_grid, opt.w_grid = np.meshgrid(
            s1_vec, w_vec, indexing='ij'
         )

        # b. utility in every grid point
        opt.u_grid = self.value_of_choice(
            opt.s1_grid,
            opt.w_grid
        )

        # c. the best point
        idx = np.unravel_index(
           np.argmax(opt.u_grid),
           opt.u_grid.shape
        )

        opt.s1 = opt.s1_grid[idx]
        opt.w = opt.w_grid[idx]

        opt.s2 = (1 - opt.s1) * opt.w
        opt.s3 = (1 - opt.s1) * (1 - opt.w)

        opt.u = opt.u_grid[idx]

        # d. results
        #opt.s1, opt.w, opt.s2, opt.s3, opt.u
        #opt.s1_grid, opt.w_grid, opt.u_grid (needed for the figures)

        return opt

    def solve(self,s0=None,do_print=True,**kwargs):
        """ solve with L-BFGS-B

        The bounds are ((0,1),(0,1)) -- the whole constraint set.

        Args:

            s0 (ndarray): starting guess for (s1,w)
            do_print (bool): print the solution
            kwargs: passed on to optimize.minimize, e.g. options={'ftol':1e-10}

        Returns:

            (SimpleNamespace): the solution and the convergence path

        """

        par = self.par
        opt = SimpleNamespace()

        # a. starting guess
        if s0 is None: s0 = np.array([0.5,0.5])
        s0 = np.asarray(s0,dtype=float)

        # b. record the path with a callback
        path = [s0.copy()]

        # c. minimize
        res = optimize.minimize(
           self.objective,
           s0,
           method='L-BFGS-B',
           bounds=((0,1),(0,1)),
           callback=lambda sk: path.append(sk.copy()),
           **kwargs
        )

        # d. results
        opt.s1 = res.x[0]
        opt.w = res.x[1]
        opt.s2 = (1 - opt.s1) * opt.w
        opt.s3 = (1 - opt.s1) * (1 - opt.w)
        opt.u = -res.fun
        opt.path = path
        opt.res = res

        return opt


Test Question 1

In [16]:
model = ConsumerClass()

grid_sol = model.solve_grid(N=500)

print("Grid solution:")
print("s1 =", grid_sol.s1)
print("w  =", grid_sol.w)
print("s2 =", grid_sol.s2)
print("s3 =", grid_sol.s3)
print("u  =", grid_sol.u)

Grid solution:
s1 = 0.5350701402805611
w  = 0.438877755511022
s2 = 0.20404737330372166
s3 = 0.26088248641571726
u  = 3.40167436585882


Compare with L-BFGS-B

In [17]:
sol = model.solve()

print("\nL-BFGS-B solution:")
print("s1 =", sol.s1)
print("w  =", sol.w)
print("s2 =", sol.s2)
print("s3 =", sol.s3)
print("u  =", sol.u)


L-BFGS-B solution:
s1 = 0.5356233806953101
w  = 0.43947902810370126
s2 = 0.20408378532610758
s3 = 0.26029283397858227
u  = 3.401679875985606


In [18]:
sol.s1 + sol.s2 + sol.s3

np.float64(1.0)

### 1.2 Calibration

In [19]:
model_sub = ConsumerClass(par={'sigma_B': 3.0})

### 1.3 How do you know your answer is right?

There is no formula for the solution of this model, so you cannot look the answer up. Two cheap checks: 

1. is the answer possible? All three shares should be strictly between 0 and 1 aand sum to one, and all three quantities should be positive

Check that the solution is possible

In [21]:
print("Budget shares:")
print("s1 =", sol.s1)
print("s2 =", sol.s2)
print("s3 =", sol.s3)

print("\nSum of shares:")
print(sol.s1 + sol.s2 + sol.s3)

print("\nQuantities:")
x1, x2, x3 = model.quantities(sol.s1, sol.w)

print("x1 =", x1)
print("x2 =", x2)
print("x3 =", x3)

Budget shares:
s1 = 0.5356233806953101
s2 = 0.20408378532610758
s3 = 0.26029283397858227

Sum of shares:
1.0

Quantities:
x1 = 5.356233806953101
x2 = 2.040837853261076
x3 = 1.7352855598572152


In [22]:
0 < sol.s1 < 1
0 < sol.s2 < 1
0 < sol.s3 < 1

np.True_

In [23]:
sol.s1 + sol.s2 + sol.s3

np.float64(1.0)

In [24]:
x1 > 0
x2 > 0
x3 > 0

np.True_

In [25]:
print("All shares between 0 and 1:",
      0 < sol.s1 < 1 and
      0 < sol.s2 < 1 and
      0 < sol.s3 < 1)

print("Shares sum to 1:",
      np.isclose(sol.s1 + sol.s2 + sol.s3, 1))

print("All quantities positive:",
      x1 > 0 and x2 > 0 and x3 > 0)

All shares between 0 and 1: True
Shares sum to 1: True
All quantities positive: True


2. Do two different methods agree? In section 2 you solve the same problem twice.

In [26]:
grid_sol = model.solve_grid(N=1000)
sol = model.solve()

# Grid search:
print("Grid search:")
print("s1 =", grid_sol.s1)
print("w  =", grid_sol.w)
print("u  =", grid_sol.u)

# L-BFGS-B optimization:
print("\nL-BFGS-B:")
print("s1 =", sol.s1)
print("w  =", sol.w)
print("u  =", sol.u)

Grid search:
s1 = 0.5355355355355356
w  = 0.4394394394394394
u  = 3.4016797975402664

L-BFGS-B:
s1 = 0.5356233806953101
w  = 0.43947902810370126
u  = 3.401679875985606


The grid search only consider a finite number of points, so its result will not necessary be identical to L-BFGS-B. But with a sufficient fine grid, the results should be almost identical. 

Calculate the difference:

In [27]:
print("Difference in s1:",
      abs(grid_sol.s1 - sol.s1))

print("Difference in w:",
      abs(grid_sol.w - sol.w))

print("Difference in utility:",
      abs(grid_sol.u - sol.u))

Difference in s1: 8.784515977455776e-05
Difference in w: 3.958866426184704e-05
Difference in utility: 7.844533955747579e-08


The solution passes both checks. All three budget shares are strictly between 0 and 1 and sum to one, which satisfies the budget constraint. The corresponding quantities of food, bus trips, and train trips are all positive. The grid search and L-BFGS-B methods produce a very similar optimal shares and utility levels. Since two independent numerical methods arrive at approximately the same solution, this provides strong evidence that the numerical solution is correct.